## 0. Path bootstrap

In [1]:
import sys
from pathlib import Path

# Ensure THIS directory is on the path so local modules are importable.
NOTEBOOK_DIR = Path().resolve()  # current working directory when running the notebook
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

print(f"Working dir : {NOTEBOOK_DIR}")

Working dir : C:\Users\vimal\OneDrive\Documents\Uni\BTP\User-Adaptive-XAI\MCC


## 1. Imports

In [2]:
import warnings

from IPython.display import display
import pandas as pd

from config import (
    # Experiment knobs
    EXPERIMENT_RESULTS_PATH,
    EXPERIMENT_TAG,
    INPUT_TEXTS,
    USER_CATEGORY,
    # XAI method  ← change XAI_METHOD in config.py to swap
    XAI_METHOD,
    XAI_NUM_FEATURES,
    XAI_NUM_SAMPLES,
    CLASS_NAMES,
    # LLM / decoding
    LAMBDA_MAP,
    NUM_BEAMS,
    USE_CONSTRAINED_DECODING,
)
from constrained_decoding import ReadabilityBeamGenerator
from model_loaders import load_classifier, load_llm
from pipeline_helpers import (
    generate_explanation,
    predict_class,
    readability_metrics,
    run_xai,
    xai_coverage,
)

warnings.filterwarnings("ignore")
print("✅ Imports complete.")

c:\Users\vimal\gpu_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Imports complete.


## 2. Experiment configuration (read-only — edit config.py)

In [3]:
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print("  EXPERIMENT CONFIG")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(f"  Tag                  : {EXPERIMENT_TAG}")
print(f"  XAI method           : {XAI_METHOD}")
print(f"  User category        : {USER_CATEGORY}")
print(f"  Constrained decoding : {USE_CONSTRAINED_DECODING}")
print(f"  Lambda value         : {LAMBDA_MAP.get(USER_CATEGORY, 'N/A')}")
print(f"  Num beams            : {NUM_BEAMS}")
print(f"  Output path          : {EXPERIMENT_RESULTS_PATH}")
n_inputs = len(INPUT_TEXTS) if INPUT_TEXTS else '(all lines in test_data.txt)'
print(f"  # Inputs             : {n_inputs}")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  EXPERIMENT CONFIG
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Tag                  : expert_lime
  XAI method           : LIME
  User category        : EXPERT
  Constrained decoding : True
  Lambda value         : 0.0
  Num beams            : 4
  Output path          : results\expert_lime.csv
  # Inputs             : 10
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


## 3. Input texts

In [4]:
# ── Resolve input list ────────────────────────────────────────────────────────
if INPUT_TEXTS is not None:
    input_texts = [t.strip() for t in INPUT_TEXTS if t.strip()]
    print(f"[Input] Using {len(input_texts)} text(s) from config.py INPUT_TEXTS.")
else:
    data_path = NOTEBOOK_DIR / "test_data.txt"
    with open(data_path, encoding="utf-8") as f:
        input_texts = [ln.strip() for ln in f if ln.strip()]
    print(f"[Input] INPUT_TEXTS is None — loaded {len(input_texts)} lines from test_data.txt.")

for i, t in enumerate(input_texts, 1):
    preview = t[:120] + ("…" if len(t) > 120 else "")
    print(f"  [{i}] ({len(t)} chars) {preview}")

[Input] Using 10 text(s) from config.py INPUT_TEXTS.
  [1] (462 chars) Endometriosis associated with massive ascites and absence of pelvic peritoneum. Although massive ascites associated with…
  [2] (670 chars) Ultrasound-Doppler diagnosis of Budd-Chiari syndrome. We report a case of apparently idiopathic Budd-Chiari syndrome, di…
  [3] (1795 chars) Neurogenic inflammation of the rat trachea: fate of neutrophils that adhere to venules. The goal of this study was to de…
  [4] (590 chars) Aberrant regeneration in a case of syringobulbia: selective co-activation of abducens and facial nerves during saccades.…
  [5] (434 chars) Germ cell tumor of testis in a patient with von Hippel-Lindau disease. Germ cell testicular tumor is a previously undesc…
  [6] (1095 chars) Failure of hepatitis B immunization in liver transplant recipients: results of a prospective trial. Twenty patients with…
  [7] (481 chars) Spontaneous rupture of an aortic aneurysm into the left renal vein. A diagnostic challe

## 4. Load models (classifier + LLM)

This is the slow step. Both models are loaded once here and reused across all inputs.

In [5]:
# ── Classifier ────────────────────────────────────────────────────────────────
clf_model, clf_pipeline = load_classifier()

# ── LLM ───────────────────────────────────────────────────────────────────────
llm_tokenizer, llm_model = load_llm()

# ── Constrained-decoding generator ────────────────────────────────────────────
generator = None
if USE_CONSTRAINED_DECODING:
    generator = ReadabilityBeamGenerator(
        model=llm_model,
        tokenizer=llm_tokenizer,
        num_beams=NUM_BEAMS,
    )
    print(f"[CD] Constrained decoding enabled — λ={LAMBDA_MAP.get(USER_CATEGORY, '?')}, beams={NUM_BEAMS}")
else:
    print("[CD] Constrained decoding disabled (greedy/sample mode).")

print("\n✅ All models loaded.")

[Loader] Loading classifier from 'C:/Users/vimal/OneDrive/Documents/Uni/BTP/User-Adaptive-XAI/Models/my_medical_model' …


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 4411.06it/s]


[Loader] Classifier ready.

[Loader] Loading LLM 'Qwen/Qwen2.5-1.5B-Instruct' …


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:02<00:00, 158.92it/s]


[Loader] LLM ready.

[CD] Constrained decoding enabled — λ=0.0, beams=4

✅ All models loaded.


## Stages 1–3 — Batch pipeline

Each input in `INPUT_TEXTS` is processed through:
- **Stage 1**: Classifier prediction + XAI feature attribution (LIME or IG)
- **Stage 2**: LLM explanation generation (input + predicted class + responsible tokens → prompt → constrained decoding)
- **Stage 3**: Readability & faithfulness metrics

In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# Batch pipeline  — Stages 1-3 repeated for every input text
# ══════════════════════════════════════════════════════════════════════════════

all_results = []  # collects one dict per input

for idx, input_text in enumerate(input_texts, start=1):
    print(f"\n{'='*64}")
    print(f"  Processing input {idx}/{len(input_texts)}")
    print(f"  Preview: {input_text[:80]}…")
    print(f"{'='*64}")

    # ── Stage 1 : Classifier prediction ────────────────────────────────────────
    print("\n[Stage 1] Classifying text …")
    predicted_class, confidence = predict_class(input_text, clf_pipeline)
    print(f"  Predicted class : {predicted_class}")
    print(f"  Confidence      : {confidence:.4f}")

    # ── Stage 1 : XAI feature attribution ──────────────────────────────────────
    print(f"\n[Stage 1] Running {XAI_METHOD} feature attribution …")
    xai_features = run_xai(
        method=XAI_METHOD,
        text=input_text,
        clf_model=clf_model,
        clf_pipeline=clf_pipeline,
        class_names=CLASS_NAMES,
        num_features=XAI_NUM_FEATURES,
        num_samples=XAI_NUM_SAMPLES,
    )
    print(f"[Stage 1] Top {len(xai_features)} {XAI_METHOD} features:")
    for word, score in xai_features:
        print(f"  {word:25s}  score={score:+.4f}")

    # ── Stage 2 : LLM explanation ───────────────────────────────────────────────
    print("\n[Stage 2] Building prompt …")
    if USE_CONSTRAINED_DECODING:
        lam = LAMBDA_MAP.get(USER_CATEGORY, LAMBDA_MAP["EXPERT"])
        print(f"[Stage 2] Generating with constrained decoding (λ={lam}, beams={NUM_BEAMS}) …")
    else:
        print("[Stage 2] Generating with standard sampling …")

    explanation = generate_explanation(
        text=input_text,
        predicted_class=predicted_class,
        xai_features=xai_features,
        user_category=USER_CATEGORY,
        tokenizer=llm_tokenizer,
        model=llm_model,
        generator=generator,
    )
    print("\n── Generated explanation ─────────────────────────────────────")
    print(explanation)
    print("─────────────────────────────────────────────────────────────")

    # ── Stage 3 : Metrics ────────────────────────────────────────────────────────
    print("\n[Stage 3] Computing metrics …")
    read_metrics = readability_metrics(explanation)
    cov = xai_coverage(explanation, xai_features)

    result = {
        "input_index":           idx,
        "experiment_tag":        EXPERIMENT_TAG,
        "xai_method":            XAI_METHOD,
        "user_category":         USER_CATEGORY,
        "constrained_decoding":  USE_CONSTRAINED_DECODING,
        "lambda":                LAMBDA_MAP.get(USER_CATEGORY, None),
        "predicted_class":       predicted_class,
        "confidence":            confidence,
        "text_snippet":          input_text[:120] + "…",
        "xai_features":          str(xai_features),
        "explanation":           explanation,
        "xai_coverage":          cov,
        **read_metrics,
    }
    all_results.append(result)
    print(f"  ✅ Input {idx} done — FRE={read_metrics.get('flesch_reading_ease', 'N/A'):.2f}  "
          f"FKGL={read_metrics.get('flesch_kincaid_grade', 'N/A'):.2f}  "
          f"coverage={cov:.2f}")

print(f"\n✅ Batch complete — {len(all_results)} input(s) processed.")


  Processing input 1/10
  Preview: Endometriosis associated with massive ascites and absence of pelvic peritoneum. …

[Stage 1] Classifying text …
  Predicted class : Digestive system diseases
  Confidence      : 0.6624

[Stage 1] Running LIME feature attribution …
[Stage 1] Top 6 LIME features:
  ascites                    score=+0.2020
  peritoneum                 score=+0.1685
  Endometriosis              score=+0.0381
  abdominal                  score=+0.0325
  destruction                score=+0.0288
  resolved                   score=+0.0271

[Stage 2] Building prompt …
[Stage 2] Generating with constrained decoding (λ=0.0, beams=4) …

── Generated explanation ─────────────────────────────────────
The model classified the biomedical abstract as "Digestive System Diseases" due to the presence of key words and phrases that are strongly indicative of this category. The most influential tokens from LIME attributions are "aspirin," "peritonitis," "diarrhea," and "appendicitis." Thes

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



── Generated explanation ─────────────────────────────────────
Based on the input and model prediction, the key tokens responsible for the prediction are "HIPPEL," "TUMOR," "DISEASE," "ENTITIES," "CELL," and "TESTICULAR." These terms are relevant because they are central to the description of the case study, which focuses on a germ cell tumor in a testicular location associated with a genetic condition (von Hippel–Lindau (VHL) disease). The VHL disease is a genetic disorder that can lead to the development of various tumors, including those in the testes. The term "germ cell tumor" specifically refers to a type of tumor that originates from germ cells, which are the cells that develop into gametes (sperm or eggs). Therefore, the model likely attributed high importance to these terms due to their direct relevance to the case being described and the specific type of cancer mentioned
─────────────────────────────────────────────────────────────

[Stage 3] Computing metrics …
  ✅ Input 5 

## 5. Results summary

In [7]:
# Summary table for all inputs
df = pd.DataFrame(all_results)

metric_cols = [
    "input_index", "xai_method", "user_category",
    "constrained_decoding", "predicted_class", "confidence",
    "flesch_reading_ease", "flesch_kincaid_grade", "smog_index",
    "xai_coverage",
]
pd.set_option("display.max_colwidth", 40)
display(df[[c for c in metric_cols if c in df.columns]])

,input_index,xai_method,user_category,constrained_decoding,predicted_class,confidence,flesch_reading_ease,flesch_kincaid_grade,smog_index,xai_coverage
0,1,LIME,EXPERT,True,Digestive system diseases,0.6624,19.542857,15.750476,16.647925,0.0000
1,2,LIME,EXPERT,True,Cardiovascular diseases,0.9097,18.975305,16.492113,16.728156,0.5000
2,3,LIME,EXPERT,True,General pathological conditions,0.5573,15.855833,17.672500,18.599290,0.1667
3,4,LIME,EXPERT,True,Nervous system diseases,0.6343,18.890915,15.717195,15.903189,0.1667
4,5,LIME,EXPERT,True,Neoplasms,0.8244,39.896150,14.302797,15.903189,1.0000
5,6,LIME,EXPERT,True,Digestive system diseases,0.6946,23.882992,14.435321,15.903189,0.5000
6,7,LIME,EXPERT,True,Cardiovascular diseases,0.7881,10.062240,16.907213,16.114345,0.5000
7,8,LIME,EXPERT,True,Neoplasms,0.8469,25.292692,14.345165,16.084391,0.3333
8,9,LIME,EXPERT,True,Neoplasms,0.4018,28.256720,14.452258,15.688483,0.3333
9,10,LIME,EXPERT,True,Cardiovascular diseases,0.4788,25.220093,16.151070,15.381576,0.6667


In [8]:
avg_fre  = df["flesch_reading_ease"].mean()
avg_fkgl = df["flesch_kincaid_grade"].mean()
avg_cov  = df["xai_coverage"].mean()

print(f"\nAverage Flesch Reading Ease    : {avg_fre:.2f}")
print(f"Average Flesch-Kincaid Grade   : {avg_fkgl:.2f}")
print(f"Average XAI Coverage           : {avg_cov:.2f}")


Average Flesch Reading Ease    : 22.59
Average Flesch-Kincaid Grade   : 15.62
Average XAI Coverage           : 0.42


## 6. Save final results

In [9]:
EXPERIMENT_RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(EXPERIMENT_RESULTS_PATH, index=False)
print(f"✅ Results saved → '{EXPERIMENT_RESULTS_PATH}' ({len(df)} row(s))")

# Print each explanation in full
for _, row in df.iterrows():
    print(f"\n── Input {int(row['input_index'])} ────────────────────────────────────────────────────")
    print(f"  Class      : {row['predicted_class']} (conf={row['confidence']:.4f})")
    print(f"  User       : {row['user_category']}")
    print(f"  Tag        : {row['experiment_tag']}")
    print("─────────────────────────────────────────────────────────────")
    print(row['explanation'])

✅ Results saved → 'results\expert_lime.csv' (10 row(s))

── Input 1 ────────────────────────────────────────────────────
  Class      : Digestive system diseases (conf=0.6624)
  User       : EXPERT
  Tag        : expert_lime
─────────────────────────────────────────────────────────────
The model classified the biomedical abstract as "Digestive System Diseases" due to the presence of key words and phrases that are strongly indicative of this category. The most influential tokens from LIME attributions are "aspirin," "peritonitis," "diarrhea," and "appendicitis." These terms are highly relevant to digestive system diseases, as they are commonly associated with conditions such as appendicitis, irritable bowel syndrome, and inflammatory bowel disease. Additionally, the mention of "peritonsillar abscess" and "pericarditis" further supports the classification, as these conditions also fall under the digestive system. The prominence of these terms in the abstract makes them crucial in determi